# NetSentinel — Expert 6: Data Exfiltration (VAE) — v3

**Unsupervised anomaly detector** for DNS exfiltration.  
Handles datasets with pre-computed numeric features OR raw DNS strings.  
Uses VAE + Isolation Forest + Mahalanobis ensemble, auto-selects best scorer.  

**Enable GPU**: Settings > Accelerator > GPU T4

In [ ]:
!pip install -q onnxruntime

In [ ]:
import os, glob, gc, json, math, time, warnings, re
from collections import Counter
from itertools import groupby
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             f1_score, accuracy_score, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', DEVICE)

EXFIL_ROOT = '/kaggle/input/cicbelldnsexf2021'
OUT_DIR = '/kaggle/working/output'
os.makedirs(OUT_DIR, exist_ok=True)

all_files = sorted(glob.glob(os.path.join(EXFIL_ROOT, '**', '*.csv'), recursive=True))
if not all_files:
    for alt in ['/kaggle/input/datasets/humera11/cicbelldnsexf2021', '/kaggle/input']:
        all_files = sorted(glob.glob(os.path.join(alt, '**', '*.csv'), recursive=True))
        if all_files:
            EXFIL_ROOT = alt
            break

print(f'Found {len(all_files)} CSV files in {EXFIL_ROOT}:')
for f in all_files:
    print(f'  {f}  ({os.path.getsize(f)/1024/1024:.1f} MB)')

## 1. Load, Label, and Inspect Columns

In [ ]:
def label_from_path(path):
    path_lower = path.lower().replace('\\', '/')
    fn = path_lower.split('/')[-1]
    if 'attack' in fn or 'exfil' in fn or 'malicious' in fn:
        return 1
    if 'benign' in fn or 'normal' in fn or 'legitimate' in fn:
        return 0
    parent = path_lower.split('/')[-2] if '/' in path_lower else ''
    if 'attack' in parent or 'exfil' in parent:
        return 1
    if parent == 'benign' or ('benign' in parent and 'attack' not in parent):
        return 0
    if 'attack' in path_lower or 'exfil' in path_lower:
        return 1
    return 0

def norm_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
    return df

# Load every CSV, record its schema signature
from collections import defaultdict
all_loaded = []
for f in all_files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df = norm_cols(df)
        label = label_from_path(f)
        tag = 'BENIGN' if label == 0 else 'EXFIL'
        print(f'  {tag:6s} | {len(df):>8,} rows | {df.shape[1]:>3} cols | {os.path.basename(f)}')
        print(f'         columns: {list(df.columns)}')
        all_loaded.append((f, df, label))
    except Exception as e:
        print(f'  ERR: {os.path.basename(f)}: {e}')

# Group by column signature
schema_groups = defaultdict(list)
for f, df, label in all_loaded:
    key = tuple(sorted(df.columns))
    schema_groups[key].append((f, df, label))

print(f'\n{len(schema_groups)} distinct schema(s) found:')
best_group = None
for key, items in schema_groups.items():
    labels_present = set(lbl for _, _, lbl in items)
    n_rows = sum(len(df) for _, df, _ in items)
    has_benign = 0 in labels_present
    has_exfil = 1 in labels_present
    print(f'  Schema ({len(key)} cols, {n_rows:,} rows): benign={has_benign}, exfil={has_exfil}')
    print(f'    Columns: {list(key)[:10]}...' if len(key) > 10 else f'    Columns: {list(key)}')
    if has_benign and has_exfil:
        if best_group is None or n_rows > sum(len(df) for _, df, _ in best_group):
            best_group = items

if best_group is None:
    # No single schema has both — try to find overlapping columns
    print('\nWARNING: No single schema has both benign and exfil.')
    print('Attempting cross-schema column intersection...')
    benign_dfs = [(f,df) for f,df,l in all_loaded if l == 0]
    exfil_dfs = [(f,df) for f,df,l in all_loaded if l == 1]
    assert benign_dfs, 'No benign files!'
    assert exfil_dfs, 'No exfil files!'
    
    # Find columns that exist in at least one benign AND one exfil file
    benign_cols = set()
    for _,df in benign_dfs:
        benign_cols |= set(df.columns)
    exfil_cols = set()
    for _,df in exfil_dfs:
        exfil_cols |= set(df.columns)
    shared = sorted(benign_cols & exfil_cols)
    print(f'  Shared columns across schemas: {shared}')
    
    if len(shared) >= 2:
        # Use shared columns
        df_benign = pd.concat([df[shared] for _,df in benign_dfs], ignore_index=True)
        df_exfil = pd.concat([df[shared] for _,df in exfil_dfs], ignore_index=True)
    else:
        # Last resort: use the schema with the most rows, treat filename as label
        print('  No shared columns! Using largest schema and relying on in-file label column')
        biggest = max(schema_groups.values(), key=lambda items: sum(len(df) for _,df,_ in items))
        df_all = pd.concat([df for _,df,_ in biggest], ignore_index=True)
        # Try to find a label column
        label_candidates = [c for c in df_all.columns if any(w in c for w in ['label','attack','class','category'])]
        print(f'  Label column candidates: {label_candidates}')
        if label_candidates:
            lc = label_candidates[0]
            vals = df_all[lc].unique()
            print(f'  Values in {lc}: {vals[:20]}')
            # Split by label column
            benign_mask = df_all[lc].astype(str).str.lower().str.contains('benign|normal|legitimate|0')
            df_benign = df_all[benign_mask].drop(columns=[lc])
            df_exfil = df_all[~benign_mask].drop(columns=[lc])
        else:
            raise ValueError('Cannot find any way to split benign/exfil!')
else:
    # Normal case: single schema with both classes
    df_benign = pd.concat([df for _,df,l in best_group if l == 0], ignore_index=True)
    df_exfil = pd.concat([df for _,df,l in best_group if l == 1], ignore_index=True)

print(f'\n>>> Benign: {len(df_benign):,} rows, {df_benign.shape[1]} cols')
print(f'>>> Exfil:  {len(df_exfil):,} rows, {df_exfil.shape[1]} cols')
print(f'>>> Columns: {list(df_benign.columns)}')
print(f'>>> Dtypes:\n{df_benign.dtypes}')

del all_loaded, schema_groups
gc.collect()


## 2. Feature Engineering (Adaptive)
If a DNS string column exists: extract 20+ character features.  
If not: use all native numeric columns directly.

In [ ]:
VOWELS = set('aeiou')

def shannon_entropy(s):
    s = str(s)
    if len(s) == 0: return 0.0
    n = len(s)
    return float(-sum((c/n)*math.log2(c/n) for c in Counter(s).values()))

def bigram_entropy(s):
    s = str(s)
    if len(s) < 2: return 0.0
    bg = [s[i:i+2] for i in range(len(s)-1)]
    n = len(bg)
    return float(-sum((c/n)*math.log2(c/n) for c in Counter(bg).values()))

def find_dns_col(df):
    for key in ['subdomain','domain','query','fqdn','hostname','host','sld','url','name']:
        for c in df.columns:
            if key in c and df[c].dtype == object:
                return c
    # Fallback: longest average string length column
    str_cols = [c for c in df.columns if df[c].dtype == object]
    if str_cols:
        avg_lens = {c: df[c].astype(str).str.len().mean() for c in str_cols}
        best = max(avg_lens, key=avg_lens.get)
        if avg_lens[best] > 5:
            return best
    return None

# EXACT column names to drop (not substrings!)
DROP_EXACT = {'label','labels','class','attack_type','category','flow_id',
              'src_ip','dst_ip','source_ip','destination_ip','timestamp',
              'attack_cat','attack_category','is_attack'}

def engineer_features(df):
    feats = {}
    dns_col = find_dns_col(df)
    n_rows = len(df)
    
    if dns_col is not None:
        print(f'  DNS string column: {dns_col!r}')
        s = df[dns_col].astype(str).fillna('')
        L = s.str.len().astype('float32')
        Lp = L + 1.0
        feats['dns_len'] = L.values
        feats['dns_log_len'] = np.log1p(L).values
        feats['dns_label_count'] = (s.str.count(r'\.') + 1).values.astype('float32')
        feats['dns_longest_token'] = s.str.split(r'[.\-_]').map(
            lambda p: max((len(x) for x in p), default=0)).values.astype('float32')
        feats['dns_entropy'] = s.map(shannon_entropy).values.astype('float32')
        feats['dns_bigram_entropy'] = s.map(bigram_entropy).values.astype('float32')
        feats['dns_norm_entropy'] = (feats['dns_entropy'] / np.log2(Lp.values + 1)).astype('float32')
        feats['dns_digit_ratio'] = (s.str.count(r'[0-9]') / Lp).values.astype('float32')
        feats['dns_upper_ratio'] = (s.str.count(r'[A-Z]') / Lp).values.astype('float32')
        feats['dns_lower_ratio'] = (s.str.count(r'[a-z]') / Lp).values.astype('float32')
        feats['dns_special_ratio'] = (s.str.count(r'[^A-Za-z0-9.]') / Lp).values.astype('float32')
        feats['dns_hex_ratio'] = (s.str.count(r'[0-9a-fA-F]') / Lp).values.astype('float32')
        feats['dns_vowel_ratio'] = s.str.lower().map(
            lambda x: sum(1 for c in x if c in VOWELS)/max(len(x),1)).values.astype('float32')
        feats['dns_unique_chars'] = s.map(lambda x: len(set(x))).values.astype('float32')
        feats['dns_unique_ratio'] = (feats['dns_unique_chars'] / Lp.values).astype('float32')
        feats['dns_max_repeat'] = s.map(
            lambda x: max((sum(1 for _ in g) for _,g in groupby(x)), default=0)).values.astype('float32')
    else:
        print('  No DNS string column found')
    
    # ALWAYS add native numeric columns (drop by EXACT name only)
    added = []
    skipped = []
    for c in df.columns:
        if c == dns_col:
            skipped.append(f'{c} (dns_col)')
            continue
        if c in DROP_EXACT:
            skipped.append(f'{c} (exact drop)')
            continue
        if c in feats:
            continue
        
        if pd.api.types.is_numeric_dtype(df[c]):
            feats[c] = pd.to_numeric(df[c], errors='coerce').values.astype('float32')
            added.append(c)
        elif df[c].dtype == object:
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().mean() > 0.5:
                feats[c] = conv.values.astype('float32')
                added.append(f'{c} (converted)')
            else:
                skipped.append(f'{c} (non-numeric object)')
    
    print(f'  Added: {added}')
    print(f'  Skipped: {skipped}')
    
    if len(feats) == 0:
        print('  EMERGENCY: 0 features! Adding ALL columns as-is')
        for c in df.columns:
            try:
                feats[c] = pd.to_numeric(df[c], errors='coerce').values.astype('float32')
            except:
                pass
    
    # Build DataFrame with explicit index to ensure correct row count
    result = pd.DataFrame(feats, index=range(n_rows))
    return result

print('=== Benign ===')
F_benign = engineer_features(df_benign)
print(f'  {F_benign.shape[1]} features, {len(F_benign):,} rows')

print('\n=== Exfil ===')
F_exfil = engineer_features(df_exfil)
print(f'  {F_exfil.shape[1]} features, {len(F_exfil):,} rows')

common_feats = sorted(set(F_benign.columns) & set(F_exfil.columns))
F_benign = F_benign[common_feats]
F_exfil = F_exfil[common_feats]
print(f'\n{len(common_feats)} common features: {common_feats}')
assert len(common_feats) >= 2, f'Only {len(common_feats)} features!'


## 3. Feature Discrimination Check

In [ ]:
n_chk = min(10000, len(F_benign), len(F_exfil))
X_chk = np.vstack([F_benign.sample(n_chk, random_state=SEED).values,
                    F_exfil.sample(n_chk, random_state=SEED).values])
X_chk = np.nan_to_num(X_chk, nan=0.0, posinf=0.0, neginf=0.0)
y_chk = np.array([0]*n_chk + [1]*n_chk)

print(f'{"Feature":40s} {"AUC":>8s} {"Signal":>8s}')
print('-'*60)
feat_scores = []
for i, c in enumerate(common_feats):
    col = X_chk[:, i]
    valid = np.isfinite(col)
    if valid.sum() < 100 or np.std(col[valid]) < 1e-12:
        continue
    auc = roc_auc_score(y_chk[valid], col[valid])
    sep = abs(auc - 0.5)
    feat_scores.append((c, auc, sep))

feat_scores.sort(key=lambda t: t[2], reverse=True)
for c, auc, sep in feat_scores:
    m = ' +++' if sep > 0.2 else ' ++' if sep > 0.1 else ' +' if sep > 0.05 else ''
    print(f'  {c:38s} {auc:8.4f} {sep:8.4f}{m}')

strong_feats = [c for c, auc, sep in feat_scores if sep > 0.02]
if len(strong_feats) < 3:
    strong_feats = [c for c, _, _ in feat_scores[:max(5, len(feat_scores))]]
    print(f'\nWARNING: weak signal, using top {len(strong_feats)} features regardless')

strong_feats = sorted(set(strong_feats))
print(f'\nSelected {len(strong_feats)} features')

## 4. Build Matrices + Split

In [ ]:
X_b = np.nan_to_num(F_benign[strong_feats].values.astype('float32'), nan=0., posinf=0., neginf=0.)
X_e = np.nan_to_num(F_exfil[strong_feats].values.astype('float32'), nan=0., posinf=0., neginf=0.)

print(f'Benign: {X_b.shape}  |  Exfil: {X_e.shape}')
assert X_b.shape[0] > 0, 'No benign rows!'
assert X_e.shape[0] > 0, 'No exfil rows!'

rng = np.random.default_rng(SEED)
idx_b = rng.permutation(len(X_b))
n1 = int(0.70 * len(X_b))
n2 = int(0.85 * len(X_b))
Xtr = X_b[idx_b[:n1]]
Xvl = X_b[idx_b[n1:n2]]
Xte_b = X_b[idx_b[n2:]]

idx_e = rng.permutation(len(X_e))
half = len(X_e) // 2
Xsel_e = X_e[idx_e[:half]]
Xte_e = X_e[idx_e[half:]]

scaler = RobustScaler().fit(Xtr)
def sc(X): return np.clip(scaler.transform(X), -10, 10).astype('float32')

Xtr_s = sc(Xtr)
Xvl_s = sc(Xvl)
Xsel_s = np.vstack([sc(Xvl), sc(Xsel_e)])
ysel = np.array([0]*len(Xvl) + [1]*len(Xsel_e))
Xte_s = np.vstack([sc(Xte_b), sc(Xte_e)])
yte = np.array([0]*len(Xte_b) + [1]*len(Xte_e))

N_FEAT = len(strong_feats)
print(f'Train: {Xtr_s.shape} | Selection: {Xsel_s.shape} | Test: {Xte_s.shape}')
print(f'Features: {N_FEAT}')

## 5. VAE Model

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_in, d_hid=256, d_lat=16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.BatchNorm1d(d_hid), nn.LeakyReLU(0.2), nn.Dropout(0.1),
            nn.Linear(d_hid, d_hid//2), nn.BatchNorm1d(d_hid//2), nn.LeakyReLU(0.2),
        )
        self.mu = nn.Linear(d_hid//2, d_lat)
        self.lv = nn.Linear(d_hid//2, d_lat)
        self.dec = nn.Sequential(
            nn.Linear(d_lat, d_hid//2), nn.BatchNorm1d(d_hid//2), nn.LeakyReLU(0.2),
            nn.Linear(d_hid//2, d_hid), nn.BatchNorm1d(d_hid), nn.LeakyReLU(0.2),
            nn.Linear(d_hid, d_in),
        )
    def forward(self, x):
        h = self.enc(x)
        mu, lv = self.mu(h), self.lv(h)
        z = mu + torch.randn_like(mu)*torch.exp(0.5*lv)
        return self.dec(z), mu, lv
    @torch.no_grad()
    def encode_mu(self, x):
        self.eval(); return self.mu(self.enc(x))
    @torch.no_grad()
    def reconstruct(self, x):
        self.eval(); return self.dec(self.mu(self.enc(x)))

def vae_loss(xr, x, mu, lv, beta=1.0):
    rec = nn.functional.mse_loss(xr, x, reduction='none').sum(1)
    kld = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1)
    return (rec + beta*kld).mean()

model = VAE(N_FEAT, d_hid=256, d_lat=16).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
loader = DataLoader(TensorDataset(torch.tensor(Xtr_s)), batch_size=512, shuffle=True)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 6. Train

In [ ]:
EPOCHS, PAT, WARMUP = 80, 12, 10
hist = {'tl':[], 'vl':[], 'auc':[]}
best_vl, best_st, wait = 1e9, None, 0
t0 = time.time()

@torch.no_grad()
def recon_err(Xs):
    model.eval(); out=[]
    for i in range(0,len(Xs),4096):
        xb=torch.tensor(Xs[i:i+4096],dtype=torch.float32,device=DEVICE)
        out.append(((model.reconstruct(xb)-xb)**2).sum(1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def get_latent(Xs):
    model.eval(); out=[]
    for i in range(0,len(Xs),4096):
        xb=torch.tensor(Xs[i:i+4096],dtype=torch.float32,device=DEVICE)
        out.append(model.encode_mu(xb).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def vloss(Xs):
    model.eval();tot=0
    for i in range(0,len(Xs),4096):
        xb=torch.tensor(Xs[i:i+4096],dtype=torch.float32,device=DEVICE)
        xr,mu,lv=model(xb);tot+=vae_loss(xr,xb,mu,lv).item()*len(xb)
    return tot/len(Xs)

for ep in range(EPOCHS):
    beta=min(1.0,(ep+1)/WARMUP)
    model.train();tot=0
    for (xb,) in loader:
        xb=xb.to(DEVICE);xr,mu,lv=model(xb);loss=vae_loss(xr,xb,mu,lv,beta=beta)
        opt.zero_grad();loss.backward();opt.step();tot+=loss.item()*len(xb)
    tl=tot/len(loader.dataset);vl=vloss(Xvl_s);sched.step(vl)
    se=recon_err(Xsel_s);auc=roc_auc_score(ysel,se)
    if auc<0.5:auc=1-auc
    hist['tl'].append(tl);hist['vl'].append(vl);hist['auc'].append(auc)
    if (ep+1)%5==0 or ep==0:
        print(f'E{ep+1:02d} TL:{tl:.4f} VL:{vl:.4f} b:{beta:.2f} AUC:{auc:.4f}')
    if vl<best_vl-1e-4:
        best_vl=vl;best_st={k:v.cpu().clone() for k,v in model.state_dict().items()};wait=0
    else:
        wait+=1
        if wait>=PAT:print(f'Early stop E{ep+1}');break

if best_st:model.load_state_dict(best_st)
train_time=time.time()-t0
print(f'Done {train_time/60:.1f}min')

## 7. Ensemble Scorers

In [ ]:
# 1) VAE recon
def score_vae(Xs): return recon_err(Xs)

# 2) Latent Mahalanobis
Z_b = get_latent(Xtr_s)
try:
    lat_cov = EmpiricalCovariance().fit(Z_b)
    def score_latent(Xs): return lat_cov.mahalanobis(get_latent(Xs))
    print('Latent Mahalanobis: OK')
except:
    score_latent = score_vae
    print('Latent Mahalanobis: fallback')

# 3) Isolation Forest
n_if = min(50000, len(Xtr_s))
iso = IsolationForest(n_estimators=300, contamination='auto', random_state=SEED, n_jobs=-1)
iso.fit(Xtr_s[rng.choice(len(Xtr_s), n_if, replace=False)])
def score_iso(Xs): return -iso.score_samples(Xs)
print('Isolation Forest: OK')

# 4) Input Mahalanobis
try:
    inp_cov = EmpiricalCovariance().fit(Xtr_s[:min(50000,len(Xtr_s))])
    def score_input_maha(Xs): return inp_cov.mahalanobis(Xs)
    print('Input Mahalanobis: OK')
except:
    score_input_maha = None
    print('Input Mahalanobis: failed')

SCORERS = {'vae_recon': score_vae, 'vae_latent': score_latent, 'iso': score_iso}
if score_input_maha: SCORERS['inp_maha'] = score_input_maha

# Evaluate each on selection set
sel_aucs = {}
for name, fn in SCORERS.items():
    raw = fn(Xsel_s)
    auc = roc_auc_score(ysel, raw)
    if auc < 0.5: auc = 1-auc
    sel_aucs[name] = auc

# Fusion
val_stats = {}
for name, fn in SCORERS.items():
    v = fn(Xvl_s); val_stats[name] = (v.mean(), v.std()+1e-9)

def score_fusion(Xs):
    zs = []
    for name, fn in SCORERS.items():
        raw=fn(Xs); m,s=val_stats[name]; z=(raw-m)/s
        zs.append(z)
    return np.mean(zs, axis=0)

fus = roc_auc_score(ysel, score_fusion(Xsel_s))
if fus<0.5: fus=1-fus
sel_aucs['fusion'] = fus

print('\nScorer ROC-AUC on selection set:')
for n,a in sorted(sel_aucs.items(), key=lambda t:-t[1]):
    print(f'  {n:15s} {a:.4f}{" <-- BEST" if a==max(sel_aucs.values()) else ""}')

BEST = max(sel_aucs, key=sel_aucs.get)
all_sc = dict(SCORERS); all_sc['fusion'] = score_fusion
anomaly_score = all_sc[BEST]
print(f'\nUsing: {BEST} (AUC={sel_aucs[BEST]:.4f})')

## 8. Test Results

In [ ]:
test_sc = anomaly_score(Xte_s)
val_sc = anomaly_score(Xvl_s)
raw_auc = roc_auc_score(yte, test_sc)
FLIP = raw_auc < 0.5
if FLIP: test_sc=-test_sc; val_sc=-val_sc; print(f'Flipped (raw={raw_auc:.4f})')

FPR = 0.01
thr = float(np.percentile(val_sc, 100*(1-FPR)))
pred = (test_sc > thr).astype(int)

pr,rc,ta = precision_recall_curve(yte, test_sc)
f1a = 2*pr*rc/(pr+rc+1e-9)
bi = np.argmax(f1a); bthr = ta[min(bi,len(ta)-1)]
pred_best = (test_sc > bthr).astype(int)

roc = roc_auc_score(yte, test_sc)
prc_auc = average_precision_score(yte, test_sc)

print(f'\n{"="*55}')
print(f'  EXPERT 6: DATA EXFILTRATION (scorer: {BEST})')
print(f'{"="*55}')
print(f'  ROC-AUC:       {roc:.4f}')
print(f'  PR-AUC:        {prc_auc:.4f}')
print(f'  Best F1:       {f1a[bi]:.4f}')
print(f'  Accuracy:      {accuracy_score(yte, pred_best):.4f}')
print(f'  Flipped:       {FLIP}')
print(f'{"="*55}')
print(classification_report(yte, pred_best, target_names=['Benign','Exfil']))

## 9. Plots

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(14,10))
ax[0,0].plot(hist['tl'],label='Train',lw=2);ax[0,0].plot(hist['vl'],label='Val',lw=2)
ax[0,0].set_title('VAE Loss',fontweight='bold');ax[0,0].legend();ax[0,0].grid(alpha=.3)
ax[0,1].plot(hist['auc'],color='green',lw=2);ax[0,1].axhline(.5,ls='--',color='gray',alpha=.5)
ax[0,1].set_title('Selection AUC',fontweight='bold');ax[0,1].set_ylim(0,1.02);ax[0,1].grid(alpha=.3)
cm=confusion_matrix(yte,pred_best)
sns.heatmap(cm,annot=True,fmt=',d',cmap='Reds',xticklabels=['B','E'],yticklabels=['B','E'],ax=ax[1,0])
ax[1,0].set_title(f'CM (F1={f1a[bi]:.4f})',fontweight='bold');ax[1,0].set_ylabel('Actual');ax[1,0].set_xlabel('Pred')
lo,hi=np.percentile(test_sc,[1,99]);bins=np.linspace(lo,hi,80)
ax[1,1].hist(test_sc[yte==0],bins=bins,alpha=.6,label='Benign',density=True,color='steelblue')
ax[1,1].hist(test_sc[yte==1],bins=bins,alpha=.6,label='Exfil',density=True,color='red')
ax[1,1].axvline(bthr,ls='--',color='k',label='Threshold');ax[1,1].set_title('Score Dist',fontweight='bold');ax[1,1].legend()
plt.suptitle(f'Expert 6: Exfiltration ({BEST})',fontsize=14,fontweight='bold')
plt.tight_layout();plt.savefig(os.path.join(OUT_DIR,'expert6_graphs.png'),dpi=150,bbox_inches='tight');plt.show()

## 10. Export

In [ ]:
import joblib
torch.save(model.cpu().state_dict(), os.path.join(OUT_DIR,'expert6_vae.pt'))

class DetR(nn.Module):
    def __init__(self,m):super().__init__();self.m=m
    def forward(self,x):return self.m.reconstruct(x)
det=DetR(model).eval();dummy=torch.randn(1,N_FEAT)
onnx_p=os.path.join(OUT_DIR,'expert6_vae.onnx')
torch.onnx.export(det,dummy,onnx_p,input_names=['features'],output_names=['recon'],
    dynamic_axes={'features':{0:'b'},'recon':{0:'b'}},opset_version=17)
print(f'ONNX: {os.path.getsize(onnx_p)/1024:.0f}KB')

import onnxruntime as ort
sess=ort.InferenceSession(onnx_p)
print(f'ONNX verify: {sess.run(None,{"features":np.random.randn(1,N_FEAT).astype(np.float32)})[0].shape}')

joblib.dump(scaler,os.path.join(OUT_DIR,'expert6_scaler.joblib'))
joblib.dump(iso,os.path.join(OUT_DIR,'expert6_iso.joblib'))

meta={'model':'Expert 6: Data Exfiltration','type':f'Unsupervised ({BEST})',
      'roc_auc':float(roc),'pr_auc':float(prc_auc),'best_f1':float(f1a[bi]),
      'accuracy':float(accuracy_score(yte,pred_best)),'flip':FLIP,
      'n_features':N_FEAT,'features':strong_feats,
      'scorer_aucs':{k:float(v) for k,v in sel_aucs.items()},
      'mitre':{'T1041':'Exfil Over C2','T1048':'Exfil Alt Protocol','T1071.004':'DNS'},
      'dataset':'CIC-Bell-DNS-EXF-2021'}
json.dump(meta,open(os.path.join(OUT_DIR,'expert6_meta.json'),'w'),indent=2)
print(f'\nROC-AUC: {roc:.4f} | PR-AUC: {prc_auc:.4f} | F1: {f1a[bi]:.4f}')

In [ ]:
import zipfile
from IPython.display import FileLink
zp='/kaggle/working/netsentinel_expert6.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT_DIR):
        fp=os.path.join(OUT_DIR,f)
        if os.path.isfile(fp):zf.write(fp,f);print(f'  {f} ({os.path.getsize(fp)/1024:.1f}KB)')
print(f'Zip: {os.path.getsize(zp)/1024/1024:.1f}MB')
FileLink(zp)